In [1]:
%pip install runwayml

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install opencv-python pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from runwayml import RunwayML

# Enter your Runway API key
client = RunwayML(
    api_key="YOUR_API_KEY"
)

# Image paths
image1 = r"C:\Users\HLBS\Desktop\pic.jpeg"
image2 = r"C:\Users\HLBS\Desktop\Priti.jpeg"
image3 = r"C:\Users\HLBS\Desktop\img3.jpeg"

# Video generation prompt
prompt = """
Create a 3-second cinematic video.
Start with Image 1, transition smoothly to Image 2,
and finish with Image 3.
Use smooth cross-dissolve transitions,
subtle zoom and camera movement.
Preserve the original image details.
"""

print("Runway API connected successfully!")
print("Image 1:", image1)
print("Image 2:", image2)
print("Image 3:", image3)
print("Video Prompt:", prompt)

Runway API connected successfully!
Image 1: C:\Users\HLBS\Desktop\pic.jpeg
Image 2: C:\Users\HLBS\Desktop\Priti.jpeg
Image 3: C:\Users\HLBS\Desktop\img3.jpeg
Video Prompt: 
Create a 3-second cinematic video.
Start with Image 1, transition smoothly to Image 2,
and finish with Image 3.
Use smooth cross-dissolve transitions,
subtle zoom and camera movement.
Preserve the original image details.



In [4]:
%pip install -U runwayml pillow requests opencv-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageOps

In [2]:


# 1. Locate the three images
def locate_image(path_without_or_with_extension):
    p = Path(path_without_or_with_extension)
    if p.is_file():
        return p
    if p.is_dir():
        candidates = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
            candidates.extend(p.glob(ext))
        if candidates:
            return sorted(candidates)[0]
    if not p.suffix:
        for ext in (".jpeg", ".jpg", ".png", ".webp"):
            candidate = p.with_suffix(ext)
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        f"Cannot find image: {path_without_or_with_extension}. "
        "Check the filename and extension."
    )

paths = [
    locate_image(r"C:\Users\HLBS\Desktop\pic.jpeg"),
    locate_image(r"C:\Users\HLBS\Desktop\Priti.jpeg"),
    locate_image(r"C:\Users\HLBS\Desktop\img3")
]
print("Images found:")
for p in paths:
    print(" -", p)

# 2. Load images and fit them to a 16:9 video frame
WIDTH, HEIGHT = 1280, 720
FPS = 30
TOTAL_SECONDS = 3
TOTAL_FRAMES = FPS * TOTAL_SECONDS

def load_and_fit(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        # Cover the video frame while preserving aspect ratio
        scale = max(WIDTH / im.width, HEIGHT / im.height)
        new_size = (round(im.width * scale), round(im.height * scale))
        im = im.resize(new_size, Image.Resampling.LANCZOS)
        left = (im.width - WIDTH) // 2
        top = (im.height - HEIGHT) // 2
        im = im.crop((left, top, left + WIDTH, top + HEIGHT))
        return np.array(im)

images = [load_and_fit(p) for p in paths]

# 3. Add a gentle zoom effect
def zoom_frame(img, progress):
    # Progress goes from 0 to 1; zoom changes subtly from 1.00x to 1.05x
    zoom = 1.0 + 0.05 * progress
    crop_w = int(WIDTH / zoom)
    crop_h = int(HEIGHT / zoom)
    x1 = (WIDTH - crop_w) // 2
    y1 = (HEIGHT - crop_h) // 2
    cropped = img[y1:y1 + crop_h, x1:x1 + crop_w]
    return cv2.resize(cropped, (WIDTH, HEIGHT), interpolation=cv2.INTER_LINEAR)

# 4. Create a 3-second sequence with two cross-fades
# Segment boundaries are at 1.0 s and 2.0 s.
# Cross-fades overlap each boundary by 0.2 s.
def frame_at(t):
    if t < 0.9:
        return zoom_frame(images[0], t / 1.0)
    elif t < 1.1:
        a = zoom_frame(images[0], min(t / 1.0, 1.0))
        b = zoom_frame(images[1], max(0.0, (t - 1.0) / 1.0))
        alpha = (t - 0.9) / 0.2
        return cv2.addWeighted(a, 1 - alpha, b, alpha, 0)
    elif t < 1.9:
        return zoom_frame(images[1], (t - 1.0) / 1.0)
    elif t < 2.1:
        a = zoom_frame(images[1], min((t - 1.0) / 1.0, 1.0))
        b = zoom_frame(images[2], max(0.0, (t - 2.0) / 1.0))
        alpha = (t - 1.9) / 0.2
        return cv2.addWeighted(a, 1 - alpha, b, alpha, 0)
    else:
        return zoom_frame(images[2], min((t - 2.0) / 1.0, 1.0))

# 5. Write the MP4 video
output_file = "Exercise3_Final_3sec.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_file, fourcc, FPS, (WIDTH, HEIGHT))

if not writer.isOpened():
    raise RuntimeError("Could not create the MP4 file. Check OpenCV installation.")

for frame_no in range(TOTAL_FRAMES):
    t = frame_no / FPS
    rgb = frame_at(t)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    writer.write(bgr)

writer.release()

# 6. Verify duration
cap = cv2.VideoCapture(output_file)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_read = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print("Video created:", Path(output_file).resolve())
print("Frames:", frame_count)
print("FPS:", fps_read)
print("Duration:", round(frame_count / fps_read, 2), "seconds")


Images found:
 - C:\Users\HLBS\Desktop\pic.jpeg
 - C:\Users\HLBS\Desktop\Priti.jpeg
 - C:\Users\HLBS\Desktop\img3.jpeg
Video created: C:\Users\HLBS\Exercise3_Final_3sec.mp4
Frames: 90
FPS: 30.0
Duration: 3.0 seconds
